# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume / CV', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/embeddinggemma-2
Updated
8 days ago
•
135k
•
2.25k
jialinyyzz/humanizer
Updated
6 days ago
•
65.6k
•
1.63k
Qwen/Qwen-Image-2.1-Turbo
Updated
3 days ago
•
12.7k
•
769
abenzerps/Qwen-Image-2.1-Uncensored-Quantized
Updated
3 days ago
•
2.32M
•
4.3k
openbmb/MiniCPM-V-4.7-35B-A3B
Up

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company in Chinese for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/embeddinggemma-2\nUpdated\n8 days ago\n•\n135k\n•\n2.25k\njialinyyzz/humanizer\nUpdated\n6 days ago\n•\n65.6k

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a pioneering AI company dedicated to building and nurturing the machine learning community of tomorrow. Their platform empowers thousands of developers, researchers, and enterprises to collaboratively create, discover, and improve machine learning models, datasets, and AI applications. By hosting over 2 million models and half a million datasets, Hugging Face serves as a vital hub for open-source AI innovation and democratization.

The company offers a rich ecosystem including **Models**, **Datasets**, **Spaces** (interactive machine learning apps), and **Buckets** (storage resources), fostering seamless collaboration between the global AI community. Their platform is the go-to destination to explore cutting-edge AI apps, share research, and experiment with state-of-the-art machine learning technologies.

---

## Core Offerings

- **Models:** Browse and utilize from over 2 million models ranging across NLP, computer vision, reinforcement learning, and more.
- **Datasets:** Access over 500,000 datasets to fuel AI training and research.
- **Spaces:** Deploy and interact with ML-powered applications built by the community.
- **Enterprise Solutions:** Hugging Face PRO offers tailored enterprise support, inference endpoints, and scalable AI infrastructure.
- **Community Tools:** Forums, blogs, Discord channels, and GitHub repositories foster active knowledge sharing and support.

---

## Company Culture

Hugging Face thrives on an open, collaborative, and community-driven culture. The company emphasizes:

- **Open-source collaboration:** The heart of Hugging Face is its vibrant open community of AI enthusiasts, researchers, and developers.
- **Knowledge sharing:** Through its blog, forums, and daily research paper highlights, it encourages ongoing education and discussion.
- **Innovation:** Hugging Face stays at the forefront of AI by integrating latest research findings, developing novel AI models, and supporting emerging technologies.
- **Ethics & Responsibility:** The community frequently discusses AI ethics, responsible development, and inclusive language models.
- **Learning & Growth:** A dynamic environment that supports learning from peers, contributing to impactful projects, and professional development.

---

## Customers and Community

Hugging Face serves a diverse range of users:

- **Researchers and Developers:** Access extensive repositories of resources and deploy models rapidly.
- **Enterprises:** Utilize scalable AI models and benefit from professional support for mission-critical applications.
- **Educators and Students:** Leverage community knowledge, datasets, and tools for teaching and research.
- **AI Enthusiasts:** Participate in forums, contribute code, and explore AI applications.

The community-driven approach fosters partnerships worldwide, demonstrated by numerous case studies and a rich collection of community articles covering cutting-edge AI topics—from NLP and computer vision to reinforcement learning and multimodal AI.

---

## Career Opportunities

Hugging Face is continuously growing and hiring passionate individuals who want to impact the future of AI. Current openings span roles in:

- Machine Learning Research and Engineering
- Software Development
- Product Management
- Community and Developer Relations
- Enterprise Sales and Support

The company offers a collaborative and innovative workplace where employees can contribute to transformative AI projects while engaging with a global community.

To explore careers and apply, visit the [Hugging Face Careers Page](https://huggingface.co/careers).

---

## Join the Hugging Face Ecosystem

Whether you are a developer, researcher, enterprise client, or AI enthusiast, Hugging Face invites you to be part of the AI revolution.

- Explore models, datasets, and apps: **https://huggingface.co/models**
- Join conversations on the [Community Forums](https://discuss.huggingface.co)
- Read insights and latest AI research on the [Blog](https://huggingface.co/blog)
- Collaborate on GitHub: **https://github.com/huggingface**
- Join their Discord community to connect with AI practitioners worldwide

---

## Contact & Connect

Website: https://huggingface.co  
Community: https://huggingface.co/community  
Careers: https://huggingface.co/careers  

---

*Hugging Face – Empowering the global AI community to build a better future, together.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is a leading AI community and collaborative platform dedicated to building the future of machine learning (ML). It serves as a central hub where ML engineers, scientists, and AI enthusiasts share, discover, and experiment with an extensive range of open-source machine learning models, datasets, and applications. With over 2 million models and half a million datasets, Hugging Face empowers the next generation of AI innovation and promotes an open, ethical AI future.

---

## What We Offer

### Collaboration and Sharing Platform  
- **Model Hub:** Explore and contribute to more than 2 million ML models across diverse tasks and domains.  
- **Datasets:** Access and share from a vast library exceeding 500,000 datasets for training and research purposes.  
- **Spaces:** Host and experiment with ML applications through user-friendly web apps integrated into the platform.  
- **Buckets:** Scalable storage solutions for your ML projects and collaborations.

### AI Products and Services  
- **HuggingChat:** An AI-powered conversational platform integrated with Hugging Face resources.  
- **Enterprise Solutions:** Scalable and secure solutions tailored for businesses, including enterprise support, inference endpoints for model deployment, and storage buckets.  
- **Hugging Face PRO:** Advanced tools and features designed for teams and organizations needing collaborative workflows and professional support.

---

## Company Culture  
At Hugging Face, openness and collaboration are core values. The company fosters a vibrant and fast-growing community of machine learning engineers, scientists, and end users invested in pushing the boundaries of AI technology. It supports ethical AI development by making cutting-edge ML tools accessible and encouraging community-driven advancements. The team is committed to innovation and transparency, fueling the open-source AI revolution with their talented science and engineering experts.

---

## Customers & Community  
Hugging Face is trusted by startups, AI researchers, and large enterprises worldwide, offering them a platform to collaborate on AI solutions and harness powerful ML tools. The community also includes educators, students, and developers benefiting from freely accessible models and datasets, which accelerate AI learning and deployment globally.

---

## Careers at Hugging Face  
Join a passionate team that is shaping the future of artificial intelligence. Hugging Face offers opportunities for roles in machine learning research, software engineering, data science, product management, and community engagement. Working here means being part of a collaborative, innovative environment that values contributions to open-source and ethical AI practices.

- Explore open positions on the Hugging Face careers page.  
- Be part of a mission-driven company dedicated to democratizing AI technology.

---

## Connect and Learn  
- **Community Platforms:** Join discussions and get support via Discord, Forum, and GitHub.  
- **Learning Resources:** Access documentation, blog posts, daily research papers, and tutorials to stay up-to-date with the latest in AI.  
- **Social Media:** Follow Hugging Face on Twitter, LinkedIn, and GitHub for news and updates.

---

## Contact & Access  
Website: [huggingface.co](https://huggingface.co)  
Community: Discord, GitHub, Forum  
Enterprise inquiries and support available via the website.

---

**Hugging Face**  
*The AI community building the future.*  
Empowering open and ethical AI for all.

---

*Brand Colors:* #FFD21E (Yellow), #FF9D00 (Orange), #6B7280 (Gray)

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face 简介手册

---

## 公司简介

Hugging Face 是一个致力于构建未来人工智能的社区平台，专注于机器学习模型、数据集和应用的协作与开发。作为全球领先的AI生态系统，Hugging Face 提供了一个开放、共享、创新的环境，聚合了超过200万款预训练模型和超过50万份数据集，致力于推动AI技术的普及和进步。

---

## 平台与产品

- **模型库（Models）**  
  超过200万款机器学习模型，涵盖自然语言处理、计算机视觉、语音识别等多个领域，支持直接浏览和部署。

- **数据集（Datasets）**  
  提供海量公开数据集，支持AI训练与测试，涵盖多种真实场景数据。

- **Spaces**  
  在线运行和分享机器学习应用的空间，无需本地配置，轻松展示创新项目和工具。

- **Buckets**  
  云端存储解决方案，便于团队数据管理与共享。

- **企业级解决方案**  
  包括Hugging Face PRO、Inference Providers和Enterprise Support，帮助企业高效部署、运维AI模型。

---

## 公司文化

Hugging Face 坚持“开放协作，推动AI进步”的理念，致力于打造包容、多元、创新的社区氛围。无论是AI研究者、工程师还是爱好者，都能在此找到支持与资源，参与社区开发、论文讨论和技术分享。官方论坛、Discord群组和GitHub仓库等多渠道精诚协作，不断推动开源AI生态的繁荣。

---

## 主要客户与合作伙伴

Hugging Face的用户包括全球科研机构、科技企业和开发者社区。多样化的模型和数据支持了从学术研究到商业应用的多重需求。企业客户通过专业版服务，获得定制化支持和专业咨询，保障AI项目高效安全落地。

---

## 职业发展机会

Hugging Face 正处于快速发展期，开放多种岗位招聘，包括但不限于机器学习科学家、软件工程师、数据工程师、产品经理及社区运营专家。加入Hugging Face，即刻参与全球AI前沿项目，享受自由开放的工作环境和极具挑战性的创新机会。

- 坚持技术与社区驱动发展
- 支持远程办公与灵活工作时间
- 丰富的学习和成长资源
- 促进多样性与包容性团队建设

如有兴趣，欢迎访问官网[https://huggingface.co](https://huggingface.co)了解更多并申请职位。

---

## 联系我们

- 网址：https://huggingface.co  
- 社区论坛、Discord及GitHub均有活跃的技术交流与支持

---

## 结语

Hugging Face 不仅是一个技术平台，更是全球机器学习社区的温床与未来AI创新的驱动力。无论您是技术爱好者，还是企业级客户，Hugging Face都为您提供了最前沿的工具和最真诚的合作机会，一同携手构建智能未来。

----

**Hugging Face，携手打造AI未来。**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>